# Base Evaluation

Generates deterministic base-model predictions before editing, then scores those predictions with the configured judge. Baseline output is saved under `data/evaluation/{model}/base/`; run-specific `perf/` materialization belongs to the normal evaluation/scoring workflow.

## Environment

In [ ]:
import copy
import gc
import json
import sys
from dataclasses import asdict
from datetime import datetime
from pathlib import Path
from statistics import mean
from typing import Any

import numpy as np
import torch
import yaml
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


LOG_DIR = PROJECT_ROOT / "data" / "evaluation" / "logs"
LOG_PATH = LOG_DIR / f"base-eval-{datetime.now():%Y-%m-%d-%H-%M-%S}.log"


def log(message: str):
    line = f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    print(line)
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as file:
        file.write(f"{line}\n")

## Evaluation Config

In [ ]:
EVAL_CONFIG_PATH = PROJECT_ROOT / "src" / "evaluate" / "config.yaml"
with open(EVAL_CONFIG_PATH, encoding="utf-8") as file:
    eval_config = yaml.safe_load(file)

model_key = eval_config["model_key"]
result_names = list(eval_config["result_names"])
judge_repeats = int(eval_config["judge"]["repeats"])
base_seed = int(eval_config["base_seed"])
generation_config = copy.deepcopy(eval_config["generation"])
judge_config = copy.deepcopy(eval_config["judge"])
judge_config["log_path"] = LOG_PATH
judge_config.setdefault("model_config", {})["seed"] = base_seed

BASE_DIR = PROJECT_ROOT / "data" / "evaluation" / model_key / "base"
set_seed(base_seed)

## Output Helpers

In [ ]:
def write_json(path: Path, payload: Any) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(f"{path.name}.tmp")
    with open(tmp_path, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)
    tmp_path.replace(path)
    return path


def read_json(path: Path) -> Any:
    with open(path, encoding="utf-8") as file:
        return json.load(file)


def base_unscored_path(result_name: str) -> Path:
    return BASE_DIR / "unscored" / f"{result_name}.json"


def base_scored_path(result_name: str) -> Path:
    return BASE_DIR / "scored" / f"j{judge_repeats}" / f"{result_name}.json"


def save_base_unscored(result_name: str, payload: dict[str, Any]) -> Path:
    return write_json(base_unscored_path(result_name), payload)


def save_base_scored(result_name: str, payload: dict[str, Any]) -> Path:
    return write_json(base_scored_path(result_name), payload)

## Data Builders

In [ ]:
from itertools import chain

from src.editors import EditRequest
from src.evaluate import EvalRecord, EvalResult, MetricResult


def load_base_dataset(result_name: str):
    if result_name not in {"es_loc", "port"}:
        raise ValueError(f"Unsupported base result: {result_name}")
    return load_dataset("Savassan/KnowledgeSurgeon", result_name, split="data")


def build_es_loc_data(record) -> tuple[EditRequest, EvalResult]:
    edit = record["edit"]
    answer = edit["answer"]
    edit_request = EditRequest(edit["subject"], edit["prompt"], answer["true"], answer["new"])

    es_records = [EvalRecord(edit["prompt"], [answer["new"]])] + [
        EvalRecord(prompt, [answer["new"]])
        for prompt in record["generality"]
    ]
    loc_records = [
        EvalRecord(item["prompt"], [item["answer"]])
        for item in record["locality"]
    ]

    eval_result = EvalResult(
        success=MetricResult(details=es_records),
        locality=MetricResult(details=loc_records),
    )
    return edit_request, eval_result


def build_port_data(record) -> tuple[EditRequest, EvalResult]:
    edit = record["edit"]
    answer = edit["answer"]
    edit_request = EditRequest(
        edit["subject"]["str"],
        edit["prompt"],
        answer["true"]["str"],
        answer["new"]["str"],
    )

    port_records = [
        EvalRecord(prompt, record["portability"]["answers"]["new"])
        for prompt in record["portability"]["prompts"]
    ]
    eval_result = EvalResult(portability=MetricResult(details=port_records))
    return edit_request, eval_result


def build_base_inputs(dataset, build_data, limit: int | None = None) -> tuple[list[EditRequest], list[EvalResult]]:
    if limit is not None and limit <= 0:
        raise ValueError("limit must be positive when set")

    selected = list(dataset)
    if limit is not None:
        if limit > len(selected):
            raise ValueError(f"Requested {limit} records, but only {len(selected)} records are available")
        selected = selected[:limit]

    built = [build_data(record) for record in selected]
    edit_requests = [item[0] for item in built]
    eval_results = [item[1] for item in built]
    return edit_requests, eval_results


def aggregate_eval_results(results: list[EvalResult]) -> EvalResult:
    if not results:
        return EvalResult()

    def aggregate_metric(metric_name: str) -> MetricResult | None:
        metric_results = [getattr(result, metric_name) for result in results]
        metric_results = [metric for metric in metric_results if metric is not None]
        if not metric_results:
            return None
        return MetricResult(
            score=mean(metric.score for metric in metric_results),
            details=list(chain.from_iterable(metric.details for metric in metric_results)),
        )

    return EvalResult(
        success=aggregate_metric("success"),
        locality=aggregate_metric("locality"),
        portability=aggregate_metric("portability"),
    )


def flatten_eval_records(eval_results: list[EvalResult]) -> list[list[EvalRecord]]:
    record_groups: list[list[EvalRecord]] = []
    for result in eval_results:
        details: list[EvalRecord] = []
        for metric_result in (result.success, result.locality, result.portability):
            if metric_result is not None:
                details.extend(metric_result.details)
        record_groups.append(details)
    return record_groups

## Base Model

In [ ]:
BASE_MODEL_CONFIG_PATH = PROJECT_ROOT / "src" / "config" / "base" / f"{model_key}.yaml"
with open(BASE_MODEL_CONFIG_PATH, encoding="utf-8") as file:
    base_model_config = yaml.safe_load(file)


def base_model_device() -> torch.device:
    gpu_id = int(base_model_config.get("gpu_id", 0))
    if torch.cuda.is_available():
        return torch.device(f"cuda:{gpu_id}")
    return torch.device("cpu")


def load_base_model_and_tokenizer():
    model_source = base_model_config["model_name"]
    fp16 = bool(base_model_config.get("fp16", True))
    model_parallel = bool(base_model_config.get("model_parallel", False))
    model_kwargs = {"dtype": torch.bfloat16 if fp16 else torch.float32}
    if model_parallel:
        model_kwargs["device_map"] = base_model_config.get("device_map") or "auto"

    log(f"[model-load-start] source={model_source} config={BASE_MODEL_CONFIG_PATH}")
    model = AutoModelForCausalLM.from_pretrained(model_source, **model_kwargs)
    if not model_parallel:
        model = model.to(base_model_device())
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(model_source)
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    log(f"[model-load-done] source={model_source}")
    return model, tokenizer

## Create Unscored Baseline

In [ ]:
from src.evaluate import generate_predictions


def get_generation_runtime_config(desc: str) -> dict[str, Any]:
    runtime_config = copy.deepcopy(generation_config)
    runtime_config.pop("async_generation", None)
    runtime_config.pop("method_overrides", None)
    runtime_config["desc"] = desc
    return runtime_config


def generate_base_unscored(
    result_name: str,
    dataset,
    build_data,
    base_model,
    base_tokenizer,
    base_limit: int | None,
    overwrite: bool,
) -> Path:
    path = base_unscored_path(result_name)
    if path.exists() and not overwrite:
        log(f"[base-unscored-skip] result={result_name} path={path}")
        return path

    edit_requests, eval_results = build_base_inputs(dataset, build_data, base_limit)
    record_groups = flatten_eval_records(eval_results)
    generation_records = sum(len(group) for group in record_groups)
    runtime_config = get_generation_runtime_config(f"Base Generation [{result_name}]")

    log(
        "[base-generation-start] "
        f"result={result_name} rows={len(eval_results)} records={generation_records} "
        f"batch_size={runtime_config.get('batch_size', 32)}"
    )
    generate_predictions(base_model, base_tokenizer, eval_results, record_groups, **runtime_config)
    result = aggregate_eval_results(eval_results)

    payload = {
        "model": model_key,
        "result_name": result_name,
        "dataset_rows": len(eval_results),
        "base_limit": base_limit,
        "seed": base_seed,
        "generation_config": runtime_config,
        "edits": [
            {
                "prompt": req.prompt,
                "old_ground_truth": req.old_ground_truth,
                "new_ground_truth": req.new_ground_truth,
            }
            for req in edit_requests
        ],
        "result": asdict(result),
    }
    saved_path = save_base_unscored(result_name, payload)
    log(f"[base-generation-done] result={result_name} path={saved_path}")
    return saved_path


def create_base_unscored(base_limit: int | None, overwrite: bool) -> list[Path]:
    builders = {
        "es_loc": build_es_loc_data,
        "port": build_port_data,
    }
    pending_results = []
    for result_name in result_names:
        if result_name not in builders:
            log(f"[skip] unsupported base result={result_name}")
        elif base_unscored_path(result_name).exists() and not overwrite:
            log(f"[base-unscored-skip] result={result_name} path={base_unscored_path(result_name)}")
        else:
            pending_results.append(result_name)

    if not pending_results:
        log("[base-unscored-skip-all] no generation work is pending")
        return []

    log(
        f"[config] action=generate loaded={EVAL_CONFIG_PATH} model={model_key} "
        f"results={pending_results}"
    )
    base_model = None
    base_tokenizer = None
    saved_paths = []
    try:
        base_model, base_tokenizer = load_base_model_and_tokenizer()
        for result_name in pending_results:
            dataset = load_base_dataset(result_name)
            saved_paths.append(
                generate_base_unscored(
                    result_name,
                    dataset,
                    builders[result_name],
                    base_model,
                    base_tokenizer,
                    base_limit=base_limit,
                    overwrite=overwrite,
                )
            )
    finally:
        base_model = None
        base_tokenizer = None
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        log("[cleanup] base model released")
    return saved_paths

## Score Baseline

In [ ]:
from src.evaluate import score_serialized_eval_result


def copy_judge_config() -> dict[str, Any]:
    return copy.deepcopy(judge_config)


async def score_base_unscored(result_name: str, overwrite: bool) -> Path:
    unscored_path = base_unscored_path(result_name)
    if not unscored_path.exists():
        raise FileNotFoundError(f"Missing baseline unscored file: {unscored_path}")

    scored_path = base_scored_path(result_name)
    if scored_path.exists() and not overwrite:
        log(f"[base-score-skip] result={result_name} judge_repeats={judge_repeats} path={scored_path}")
        return scored_path

    unscored_payload = read_json(unscored_path)
    repeat_config = copy_judge_config()
    log(f"[base-score-start] result={result_name} judge_repeats={judge_repeats}")
    scored_result = await score_serialized_eval_result(
        copy.deepcopy(unscored_payload["result"]),
        **repeat_config,
    )
    scored_payload = {
        **{key: value for key, value in unscored_payload.items() if key != "result"},
        "judge_repeats": judge_repeats,
        "result": scored_result,
    }
    saved_path = save_base_scored(result_name, scored_payload)
    log(f"[base-score-done] result={result_name} judge_repeats={judge_repeats} path={saved_path}")
    return saved_path


async def score_base_results(overwrite: bool) -> list[Path]:
    log(
        f"[config] action=score loaded={EVAL_CONFIG_PATH} model={model_key} "
        f"results={result_names} judge_repeats={judge_repeats}"
    )
    saved_paths = []
    for result_name in result_names:
        if result_name not in {"es_loc", "port"}:
            log(f"[skip] unsupported base result={result_name}")
            continue
        saved_paths.append(await score_base_unscored(result_name, overwrite=overwrite))
    return saved_paths

## Behavior Configuration

In [ ]:
# Full-dataset generation is reusable across arbitrary benchmark samples.
# Use a small positive integer only for a smoke test.
BASE_LIMIT: int | None = None
OVERWRITE_UNSCORED = False
OVERWRITE_SCORED = False

## Manual Actions

Uncomment one action at a time.

In [ ]:
# Generate base-model predictions. This loads and releases the base model internally.
# create_base_unscored(
#     base_limit=BASE_LIMIT,
#     overwrite=OVERWRITE_UNSCORED,
# )

# Score existing unscored base results without loading the base model or datasets.
# await score_base_results(overwrite=OVERWRITE_SCORED)